In [ ]:
# === Setup (T4 expected; guard for P100 sm_60) ===
import subprocess, sys
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("Assigned GPU:", gpu, flush=True)
if "P100" in gpu:
    print("P100 -> installing sm_60-compatible torch (cu121)...", flush=True)
    subprocess.run([sys.executable,"-m","pip","install","-q",
        "torch==2.5.1","torchvision==0.20.1",
        "--index-url","https://download.pytorch.org/whl/cu121"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics"], check=False)

import os, glob, csv, hashlib
from pathlib import Path
import yaml, torch
from ultralytics import YOLO
_x = torch.zeros(1, device="cuda"); assert float((_x+1).cpu()) == 1.0
print("Torch:", torch.__version__, "| CUDA OK on", torch.cuda.get_device_name(0), flush=True)

NC = 22
NAMES = [
    "bean_angular_leaf_spot","bean_bean_rust","bean_healthy",
    "cauliflower_bacterial_spot_rot","cauliflower_black_rot",
    "cauliflower_downy_mildew","cauliflower_no_disease",
    "chilli_Curling-Leafs","chilli_Healthy-Leafs","chilli_Leaf Spot-Leafs",
    "chilli_Leveillula taurica-Leafs","chilli_Mozaik Leaf",
    "cucumber_Antraknosa","cucumber_Bacterial Wilt","cucumber_Downy Mildew",
    "potato_healthy","potato_blackleg","potato_late_blight","potato_early_blight",
    "tomato_bacterial_spot","tomato_curl_virus","tomato_two_spotted_spider_mite",
]
GROUP_B_LESION = {
    "potato_healthy","cucumber_Antraknosa","cucumber_Bacterial Wilt",
    "potato_late_blight","potato_early_blight","bean_bean_rust",
    "potato_blackleg","bean_angular_leaf_spot",
}
group_of = lambda c: "B_lesion" if c in GROUP_B_LESION else "A_whole_leaf"


In [ ]:
# === Locate v12 and rewrite data.yaml for the Kaggle mount ===
cands = [p for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True) if "v12" in p]
assert cands, "merged-dataset-v12 not found under /kaggle/input"
V12 = str(Path(cands[0]).parent); print("V12 root:", V12)
v12_yaml = "/kaggle/working/data_v12.yaml"
yaml.safe_dump({"path": V12, "train": "images/train", "val": "images/valid",
                "test": "images/test", "nc": NC, "names": NAMES},
               open(v12_yaml, "w"), sort_keys=False)
for s in ["train","valid","test"]:
    print(f"  {s:<6} {len(glob.glob(f'{V12}/images/{s}/*')):>6} images")


In [ ]:
# === Leakage guard ===
def md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p,"rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def hashes_of(d): return {md5(p) for p in glob.glob(d + "/*")}
tr = hashes_of(f"{V12}/images/train")
lv = len(tr & hashes_of(f"{V12}/images/valid"))
lt = len(tr & hashes_of(f"{V12}/images/test"))
print(f"leak train<->valid: {lv} | train<->test: {lt}")
assert lv == 0 and lt == 0, "LEAKAGE - stop"
print("v12 leakage-free.")


In [ ]:
# === Per-class validation helper ===
def val_perclass(weights, data_yaml, tag, split="val"):
    b = YOLO(weights).val(data=data_yaml, split=split, imgsz=640, verbose=False).box
    rows = []
    for i, ci in enumerate(b.ap_class_index):
        ci = int(ci); name = NAMES[ci] if ci < len(NAMES) else str(ci)
        rows.append([tag, name, group_of(name),
                     round(float(b.p[i]),4), round(float(b.r[i]),4),
                     round(float(b.ap50[i]),4), round(float(b.ap[i]),4)])
    rows.append([tag, "ALL", "ALL", round(float(b.mp),4), round(float(b.mr),4),
                 round(float(b.map50),4), round(float(b.map),4)])
    return rows


In [ ]:
# === Train yolo11n.pt on clean v12 - CONTROLLED 60-epoch budget ===
# NO time= parameter: it replaces the epoch schedule and rescales in both directions
# (that is what pushed yolo11s to 122 ep and cut yolo11m to 60 ep in the earlier runs).
# patience=100 > epochs=60 so early stopping can never fire -> all three models get an
# IDENTICAL 60-epoch budget. best.pt still tracks the best epoch.
RUN = "nano60"
model = YOLO("yolo11n.pt")
model.train(data=v12_yaml, epochs=60, imgsz=640, batch=16, device=0,
            amp=True, patience=100, seed=0,
            project="/kaggle/working/runs", name=RUN)


In [ ]:
# === Validate on clean v12 val set ===
rows = val_perclass(f"/kaggle/working/runs/{RUN}/weights/best.pt", v12_yaml, "v12_nano_60ep")
out_csv = "/kaggle/working/perclass_v12_nano60.csv"
with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["arm","class","group","precision","recall","mAP50","mAP50_95"])
    w.writerows(rows)
print("SAVED", out_csv)


In [ ]:
# === Report ===
import pandas as pd
df = pd.read_csv(f"/kaggle/working/runs/{RUN}/results.csv")
df.columns = [c.strip() for c in df.columns]
n_ep = len(df)
wall = float(df["time"].iloc[-1]) / 3600 if "time" in df.columns else float("nan")
mcol = [c for c in df.columns if "mAP50(B)" in c and "95" not in c][0]
print(f"epochs completed: {n_ep}/60  wall: {wall:.2f}h")
assert n_ep == 60, f"EXPECTED EXACTLY 60 EPOCHS, GOT {n_ep} - budget not controlled"
print(f"best mAP50 during training: {df[mcol].max():.4f} @ epoch {int(df[mcol].idxmax())+1}")
for g in ("A_whole_leaf","B_lesion"):
    sel = [r for r in rows if r[2] == g]
    if sel:
        print(f"{g}: n={len(sel)} mAP50={sum(r[5] for r in sel)/len(sel):.3f} "
              f"recall={sum(r[4] for r in sel)/len(sel):.3f}")
allr = [r for r in rows if r[1]=="ALL"][0]
print(f"ALL: mAP50={allr[5]:.3f} recall={allr[4]:.3f}")
print()
print("CONFOUNDED earlier runs (unequal budgets - do NOT compare against these):")
print("  n 100ep=0.8975 | s 122ep=0.9135 | m 60ep(time-capped)=0.9035")
print()
for r in rows: print(r)
